# Import Basic Libraries and Modules

In [ ]:
# Loading Libraries
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pickle
import os
import time
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.utils import resample
import shap
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns



# Inputs & Parameters

In [ ]:

# File paths and other parameters
Nbins= 2
property= 'Activity'
# Names of files that contain SMILES, composition, and target property 
smiles_file_training = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Data\Data_Combined\{property}\InVitro_SMILES_Training.xlsx"
smiles_file_testing = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Data\Data_Combined\{property}\InVitro_SMILES_Testing.xlsx"
smiles_sheet_name= "SMILES"

target_file_training = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Data\Data_Combined\{property}\InVitro_{property}_Training.xlsx"
target_file_testing = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Data\Data_Combined\{property}\InVitro_{property}_Testing.xlsx"
target_sheet_name= f"{Nbins} bins"

output_file = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\Feature_Importance\{property}_Feature_Importance.xlsx"

target_style= 'NG'
featurizer_style= 'All_Featurizers_'+target_style
ml_model_style= 'All_Models_'+target_style


# Implementation >>>>

# Load pickle files

In [ ]:

training_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\datay_training_{Nbins}bins_{target_style}.pkl"
testing_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\datay_testing_{Nbins}bins_{target_style}.pkl"


with open(training_filepath, 'rb') as file1:
    datay_training= pickle.load(file1)
with open(testing_filepath, 'rb') as file2:
    datay_testing= pickle.load(file2)


training_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\dataX_dict_training_{featurizer_style}.pkl"
testing_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\dataX_dict_testing_{featurizer_style}.pkl"
with open(training_filepath, 'rb') as file1:
    print('Reading training featurized variable')
    dataX_dict_training= pickle.load(file1)
with open(testing_filepath, 'rb') as file2:
        print('Reading testing featurized variable')
        dataX_dict_testing= pickle.load(file2)




# Get names of RDKit descriptors

In [ ]:
RDKit_descriptor_names = [name for name, _ in Descriptors._descList if name != "Ipc"]
RDKit_descriptor_names= RDKit_descriptor_names[:-1]

RDKit_descriptor_names_all = []
for i in range(1, 5):  # For constituents 1 to 4
    RDKit_descriptor_names_all.extend([f"{name} {i}" for name in RDKit_descriptor_names])


# Add extra descriptors
extra_descriptors = ['Composition 1', 'Composition 2', 'Composition 3', 
                     'Composition 4', 'RNA type', 'Lipid to RNA', 'Dosage']
RDKit_descriptor_names_all.extend(extra_descriptors)

print(len(RDKit_descriptor_names_all))

# Permutation Importance

In [ ]:

# featurizer_name= 'RDKit_Descriptors_NG'
# # Iterate over the featurizer functions and ml model functions
# with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists= 'replace') as writer:

#     dataX_training= dataX_dict_training[featurizer_name]
#     dataX_testing= dataX_dict_testing[featurizer_name]

#     model = RandomForestClassifier(n_estimators= 100, max_depth= 10, min_samples_leaf= 4, min_samples_split= 10, random_state= 42, class_weight= 'balanced')
#     model.fit(dataX_training, datay_training)
 
#     num_features = dataX_training.shape[1]
#     feature_id = [i for i in range(num_features)]
    
#     perm_importance = permutation_importance(model, dataX_testing, datay_testing, n_repeats=10, random_state=42)
#     perm_importance_mean= perm_importance.importances_mean
#     perm_importance_df = pd.DataFrame({
#         'Feature ID': feature_id,
#         'Feature Name': RDKit_descriptor_names_all,
#         'Feature Importance': perm_importance_mean
#         })
#     perm_importance_sorted = perm_importance_df.sort_values(by='Feature Importance', ascending=False)
#     perm_importance_df.to_excel(writer, sheet_name='Perm_'+featurizer_name, index=False)

# print(f"Finished")

# Get Metrics using Specific Number of Top Features based on Permutation Importance

In [ ]:
# num_top_sheet_name= 'Perm_Num_Top'
# featurizer_name= 'RDKit_Descriptors_NG'

# dataX_training= dataX_dict_training[featurizer_name]
# dataX_testing= dataX_dict_testing[featurizer_name]

# num_top_descriptors = [10, 25, 40, 45, 50, 55, 60, 75, 100, 200, 300, 400]

# # Initialize an empty dictionary to store the accuracies
# accuracy_dict = {}
# precision_dict = {}
# recall_dict = {}
# f1score_dict = {}

# for num_top in num_top_descriptors:
#     # Select top features based on feature importance
#     num_top_name= 'Top_'+ str(num_top)
#     top_indices = perm_importance_sorted.head(num_top)['Feature ID'].values
#     dataX_training_new = dataX_training[:, top_indices]
#     dataX_testing_new = dataX_testing[:, top_indices]

#     # Predictions using RF classification
#     predicted_class = ML_Model_RF(dataX_training_new, dataX_testing_new, datay_training, datay_testing)
#     true_class = datay_testing

#     classes = np.unique(true_class)
#     accuracy_per_class = {}
#     precision_per_class = {}
#     recall_per_class = {}
#     f1score_per_class = {}
#     for cls in classes:
#         # Binary classification: 1 for current class, 0 for all other classes
#         y_true_binary = [1 if y == cls else 0 for y in true_class]
#         y_pred_binary = [1 if y == cls else 0 for y in predicted_class]
#         accuracy = accuracy_score(y_true_binary, y_pred_binary)
#         precision = precision_score(y_true_binary, y_pred_binary)
#         recall = recall_score(y_true_binary, y_pred_binary)
#         f1score = f1_score(y_true_binary, y_pred_binary)
        
#         accuracy_per_class[cls] = accuracy
#         precision_per_class[cls] = precision
#         recall_per_class[cls] = recall
#         f1score_per_class[cls] = f1score
        
#     accuracy_dict[num_top_name] = accuracy_per_class
#     precision_dict[num_top_name] = precision_per_class
#     recall_dict[num_top_name] = recall_per_class
#     f1score_dict[num_top_name] = f1score_per_class


# # Prepare the data for the DataFrame
# rows = []

# for num_top in num_top_descriptors:
#     num_top_name= 'Top_'+ str(num_top)
#     # Initialize the row with model and featurizer names
#     row = {'Num Top Descriptors': num_top_name}

#     # Add accuracy for each class
#     for cls in classes:
#         row[f'Accuracy_{cls}'] = accuracy_dict[num_top_name].get(cls, None)
#     # Add precision for each class
#     for cls in classes:
#         row[f'Precision_{cls}'] = precision_dict[num_top_name].get(cls, None)
#     # Add recall for each class
#     for cls in classes:
#         row[f'Recall_{cls}'] = recall_dict[num_top_name].get(cls, None)
#     # Add f1 score for each class
#     for cls in classes:
#         row[f'F1 Score_{cls}'] = f1score_dict[num_top_name].get(cls, None)

#     # Append the row to the list
#     rows.append(row)

# # Convert to DataFrame
# results_df = pd.DataFrame(rows)

# # Write DataFrame to Excel
# # Check if the output file exists and read existing data if it does
# if os.path.exists(output_file):
#     with pd.ExcelFile(output_file, engine='openpyxl') as xls:
#         if num_top_sheet_name in xls.sheet_names:
#             existing_data_df = pd.read_excel(xls, sheet_name=num_top_sheet_name)
#             # Concatenate existing data with new data
#             combined_df = pd.concat([existing_data_df, results_df], ignore_index=True)
#         else:
#             combined_df = results_df
# else:
#     combined_df = results_df

# # Write the combined DataFrame to the Excel file
# with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     combined_df.to_excel(writer, sheet_name=num_top_sheet_name, index=False)

# print(f"Finished")


# SHAP Test

In [ ]:

featurizer_name= 'RDKit_Descriptors_NG'
dataX_training= dataX_dict_training[featurizer_name]
dataX_testing= dataX_dict_testing[featurizer_name]
num_features = dataX_training.shape[1]
feature_id = [i for i in range(num_features)]
feature_names = RDKit_descriptor_names_all

model = RandomForestClassifier(n_estimators= 100, max_depth= 10, min_samples_leaf= 4, min_samples_split= 10, random_state= 42, class_weight= 'balanced')
model.fit(dataX_training, datay_training)



# Explain predictions
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(dataX_testing)

# Calculate Interaction Values
interaction_values_all = explainer.shap_interaction_values(dataX_testing)

interaction_values = interaction_values_all[1]
interaction_list = []
for i in range(num_features):
    for j in range(i+1, num_features):  # Avoid double counting
        interaction_importance = np.abs(interaction_values[:, i, j]).mean()
        interaction_list.append((feature_names[i], feature_names[j], interaction_importance))

interaction_df = pd.DataFrame(interaction_list, columns=['Feature 1', 'Feature 2', 'Interaction Importance'])
interaction_df_sorted = interaction_df.sort_values(by='Interaction Importance', ascending=False)


# Compute average SHAP values (absolute importance)
shap_importance = np.abs(shap_values[1]).mean(axis=0)
shap_importance_df = pd.DataFrame({'Feature ID': feature_id, 'Feature Name': RDKit_descriptor_names_all, 'SHAP Importance': shap_importance})
shap_importance_sorted = shap_importance_df.sort_values(by='SHAP Importance', ascending=False)
    
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists= 'replace') as writer:    
    shap_importance_df.to_excel(writer, sheet_name='SHAP_'+featurizer_name, index=False)
    interaction_df_sorted.to_excel(writer, sheet_name='SHAP Interactions_'+featurizer_name, index=False)

print(f"Finished")



In [ ]:

# Bar Plot
plt.figure(figsize=(10, 8))
sns.barplot(x='SHAP Importance', y='Feature Name', data=shap_importance_sorted[:20], palette='coolwarm')
plt.title('Top 20 Features by SHAP Importance')
plt.xlabel('Mean Absolute SHAP Value')
plt.ylabel('Feature Name')
plt.show()


In [ ]:


# Pivot DataFrame to create heatmap
interaction_pivot = interaction_df_sorted[0:100].pivot(index='Feature 1', columns='Feature 2', values='Interaction Importance')

plt.figure(figsize=(12, 10))
sns.heatmap(interaction_pivot, cmap='coolwarm', annot=False, fmt='.2f', linewidths=0.5)
plt.title('Shapley Interaction Index - Feature Interactions')
plt.show()


In [ ]:
# Create Network Graph
G = nx.Graph()

# Only show interactions above a threshold
threshold = 0.0005
filtered_interactions = interaction_df_sorted[interaction_df_sorted['Interaction Importance'] > threshold]

# Add nodes and edges
for _, row in filtered_interactions.iterrows():
    G.add_node(row['Feature 1'])
    G.add_node(row['Feature 2'])
    G.add_edge(row['Feature 1'], row['Feature 2'], weight=row['Interaction Importance'])

# Visualization
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
edges = G.edges(data=True)

nx.draw_networkx_nodes(G, pos, node_size=800, node_color='skyblue')
nx.draw_networkx_labels(G, pos, font_size=10)

edge_weights = [d['weight'] for _, _, d in edges]
nx.draw_networkx_edges(G, pos, edgelist=edges, width=edge_weights, edge_color=edge_weights, edge_cmap=plt.cm.coolwarm)

plt.title('Feature Interaction Network (Shapley Interaction Index)')
plt.show()


In [ ]:
# Define the indices of the descriptors you want to plot
# selected_indices = [11, 3, 842, 24, 51, 23, 20, 57, 74, 22, 99, 0, 21, 45, 28, 65, 15, 41, 8, 46, 77, 44, 73, 43, 4, 19, 96, 1, 42, 14, 66, 82, 87, 30, 32, 40, 105, 6, 86, 34, 18, 836, 36, 89, 122, 84, 33, 837]
selected_indices = [11, 3, 842, 24, 51, 23, 20, 57, 74, 22, 99, 0]

# 1. Subset SHAP values: Select specific descriptors for all LNPs
shap_values_selected = shap_values[1][:, selected_indices]

# 2. Subset Data: Select specific descriptors from the dataset
dataX_testing_selected = dataX_testing[:, selected_indices]

# 3. Subset Descriptor Names: Select specific descriptor names
RDKit_descriptor_names_selected = [RDKit_descriptor_names_all[i] for i in selected_indices]

# 4. Generate SHAP Violin Plot (Layered Violin)
import shap
shap.plots.violin(
    shap_values_selected, 
    features=dataX_testing_selected, 
    feature_names=RDKit_descriptor_names_selected, 
    plot_type="layered_violin", 
    layered_violin_max_num_bins=20, 
    max_display=len(selected_indices)
)


In [ ]:
shap.summary_plot(shap_values[1], dataX_testing, feature_names=RDKit_descriptor_names_all, plot_type="violin", max_display=10)

In [ ]:
shap.summary_plot(shap_values[1], features=dataX_testing, feature_names=RDKit_descriptor_names_all, max_display=20, cmap= "coolwarm")

In [ ]:
shap.dependence_plot("MaxPartialCharge 1", shap_values[1], dataX_testing, feature_names=RDKit_descriptor_names_all)

shap.dependence_plot("MaxPartialCharge 1", shap_values=shap_values[1], features=dataX_testing, feature_names=RDKit_descriptor_names_all, interaction_index="Dosage")

# Get Metrics using Specific Number of Top Features based on SHAP Importance

In [ ]:
# num_top_sheet_name= 'SHAP_Num_Top'
# featurizer_name= 'RDKit_Descriptors_NG'

# dataX_training= dataX_dict_training[featurizer_name]
# dataX_testing= dataX_dict_testing[featurizer_name]

# num_top_descriptors = [10, 25, 40, 45, 50, 55, 60, 75, 100, 200, 300, 400]

# # Initialize an empty dictionary to store the accuracies
# accuracy_dict = {}
# precision_dict = {}
# recall_dict = {}
# f1score_dict = {}

# for num_top in num_top_descriptors:
#     # Select top features based on feature importance
#     num_top_name= 'Top_'+ str(num_top)
#     top_indices = shap_importance_sorted.head(num_top)['Feature ID'].values
#     dataX_training_new = dataX_training[:, top_indices]
#     dataX_testing_new = dataX_testing[:, top_indices]

#     # Predictions using RF classification
#     predicted_class = ML_Model_RF(dataX_training_new, dataX_testing_new, datay_training, datay_testing)
#     true_class = datay_testing

#     classes = np.unique(true_class)
#     accuracy_per_class = {}
#     precision_per_class = {}
#     recall_per_class = {}
#     f1score_per_class = {}
#     for cls in classes:
#         # Binary classification: 1 for current class, 0 for all other classes
#         y_true_binary = [1 if y == cls else 0 for y in true_class]
#         y_pred_binary = [1 if y == cls else 0 for y in predicted_class]
#         accuracy = accuracy_score(y_true_binary, y_pred_binary)
#         precision = precision_score(y_true_binary, y_pred_binary)
#         recall = recall_score(y_true_binary, y_pred_binary)
#         f1score = f1_score(y_true_binary, y_pred_binary)
        
#         accuracy_per_class[cls] = accuracy
#         precision_per_class[cls] = precision
#         recall_per_class[cls] = recall
#         f1score_per_class[cls] = f1score
        
#     accuracy_dict[num_top_name] = accuracy_per_class
#     precision_dict[num_top_name] = precision_per_class
#     recall_dict[num_top_name] = recall_per_class
#     f1score_dict[num_top_name] = f1score_per_class


# # Prepare the data for the DataFrame
# rows = []

# for num_top in num_top_descriptors:
#     num_top_name= 'Top_'+ str(num_top)
#     # Initialize the row with model and featurizer names
#     row = {'Num Top Descriptors': num_top_name}

#     # Add accuracy for each class
#     for cls in classes:
#         row[f'Accuracy_{cls}'] = accuracy_dict[num_top_name].get(cls, None)
#     # Add precision for each class
#     for cls in classes:
#         row[f'Precision_{cls}'] = precision_dict[num_top_name].get(cls, None)
#     # Add recall for each class
#     for cls in classes:
#         row[f'Recall_{cls}'] = recall_dict[num_top_name].get(cls, None)
#     # Add f1 score for each class
#     for cls in classes:
#         row[f'F1 Score_{cls}'] = f1score_dict[num_top_name].get(cls, None)

#     # Append the row to the list
#     rows.append(row)

# # Convert to DataFrame
# results_df = pd.DataFrame(rows)

# # Write DataFrame to Excel
# # Check if the output file exists and read existing data if it does
# if os.path.exists(output_file):
#     with pd.ExcelFile(output_file, engine='openpyxl') as xls:
#         if num_top_sheet_name in xls.sheet_names:
#             existing_data_df = pd.read_excel(xls, sheet_name=num_top_sheet_name)
#             # Concatenate existing data with new data
#             combined_df = pd.concat([existing_data_df, results_df], ignore_index=True)
#         else:
#             combined_df = results_df
# else:
#     combined_df = results_df

# # Write the combined DataFrame to the Excel file
# with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     combined_df.to_excel(writer, sheet_name=num_top_sheet_name, index=False)

# print(f"Finished")


# H-statistics

In [ ]:

def calculate_h_statistic(model, X, feature_indices):
    """
    Calculate the H-statistic for interaction between two features.
    H = 1 - (independent_effect / combined_effect)
    """
    # Calculate Partial Dependence for individual features
    pdp_feature1 = PartialDependenceDisplay.from_estimator(model, X, [feature_indices[0]], grid_resolution=50, kind='average')
    pdp_feature2 = PartialDependenceDisplay.from_estimator(model, X, [feature_indices[1]], grid_resolution=50, kind='average')
    pdp_combined = PartialDependenceDisplay.from_estimator(model, X, [feature_indices], grid_resolution=50, kind='average')
    
    independent_effect = np.add(pdp_feature1.pd_results[0]['average'], pdp_feature2.pd_results[0]['average'])
    combined_effect = pdp_combined.pd_results[0]['average']

    # Calculate H-statistic
    h_statistic = 1 - np.var(combined_effect - independent_effect) / np.var(combined_effect)
    return h_statistic

# Calculate H-Statistic for Selected Feature Pairs
h_stats = []
for i in range(num_features):
    for j in range(i + 1, num_features):
        h_value = calculate_h_statistic(model, dataX_testing, [i, j])
        h_stats.append((feature_names[i], feature_names[j], h_value))

# Store Results in DataFrame
h_stat_df = pd.DataFrame(h_stats, columns=['Feature 1', 'Feature 2', 'H-Statistic'])
h_stat_df_sorted = h_stat_df.sort_values(by='H-Statistic', ascending=False)

# Save H-Statistic Results to Excel
output_file = "H_Statistic_Feature_Interactions.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl', mode='w') as writer:
    h_stat_df_sorted.to_excel(writer, sheet_name='Feature Interaction Strength', index=False)

print("✅ H-statistic calculated and saved.")


# Visualizing H-statistic results

In [ ]:


# Pivot DataFrame for Heatmap
h_stat_pivot = h_stat_df_sorted.pivot(index='Feature 1', columns='Feature 2', values='H-Statistic')

plt.figure(figsize=(12, 10))
sns.heatmap(h_stat_pivot, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5)
plt.title('H-Statistic - Feature Interaction Strength')
plt.show()


In [ ]:


# Select Top Interactions
top_interactions = h_stat_df_sorted.sort_values(by='H-Statistic', ascending=False).head(5)

for _, row in top_interactions.iterrows():
    feature_1 = row['Feature 1']
    feature_2 = row['Feature 2']
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=dataX_testing[:, RDKit_descriptor_names_all.index(feature_1)], 
                    y=dataX_testing[:, RDKit_descriptor_names_all.index(feature_2)], 
                    hue=datay_testing, palette='coolwarm')
    plt.title(f'Interaction: {feature_1} vs. {feature_2} (H = {row["H-Statistic"]:.2f})')
    plt.xlabel(feature_1)
    plt.ylabel(feature_2)
    plt.show()


# LIME Test

In [ ]:


# featurizer_style= 'RDKit_Descriptors_NG'
# # Iterate over the featurizer functions and ml model functions
# with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists= 'replace') as writer:
#     for ff in featurizer_map[featurizer_style]:
#         featurizer_name = ff.__name__  # Get the featurizer function name
    
#         dataX_training= dataX_dict_training[featurizer_name]
#         dataX_testing= dataX_dict_testing[featurizer_name]
    
#         model = RandomForestClassifier(n_estimators= 100, max_depth= 10, min_samples_leaf= 4, min_samples_split= 10, random_state= 42, class_weight= 'balanced')
#         model.fit(dataX_training, datay_training)
     
#         num_features = dataX_training.shape[1]
#         feature_id = [i for i in range(num_features)]
        
#         # Explain predictions
#         explainer = lime.lime_tabular.LimeTabularExplainer(dataX_training, 
#                                                    feature_names=RDKit_descriptor_names, 
#                                                    class_names=['0', '1'], 
#                                                    discretize_continuous=True)

# print(f"Finished")


# # Explain a single prediction
# instance_idx = 0
# exp = explainer.explain_instance(dataX_testing[instance_idx], model.predict_proba)
# exp.show_in_notebook()

# # # Save explanation
# # exp.save_to_file('lime_explanation.html')


# Write top 100 feature values to file

In [ ]:
num_estimators= 200
num_neighbors= 5
num_random= 42
num_top= 100

dataX_train= dataX_train_dict['RDKit_Descriptors_Featurizer']
datay_train= datay_train_dict['RDKit_Descriptors_Featurizer']
dataX_test= dataX_test_dict['RDKit_Descriptors_Featurizer']
datay_test= datay_test_dict['RDKit_Descriptors_Featurizer']

num_features = dataX_train.shape[1]
feature_id = [i for i in range(num_features)]
model = RandomForestClassifier(n_estimators= num_estimators, random_state= num_random)
model.fit(dataX_train, datay_train)
importances = model.feature_importances_
feature_importances = pd.DataFrame({
'Feature ID': feature_id,
'Feature Importance': importances
})
feature_importances = feature_importances.sort_values(by='Feature Importance', ascending=False)
top_indices = feature_importances.head(num_top)['Feature ID'].values

# Create a DataFrame for datay_test
df_datay_test = pd.DataFrame(datay_test, columns=['True Class'])
# Create a DataFrame for dataX_test[:, top_indices]
df_dataX_test_top = pd.DataFrame(dataX_test[:, top_indices], columns=[f'Feature {i+1}' for i in top_indices])
# Concatenate both DataFrames along columns
df_combined = pd.concat([df_datay_test, df_dataX_test_top], axis=1)

# Write the combined DataFrame to an Excel file
with pd.ExcelWriter(output_file, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_combined.to_excel(writer, sheet_name=f'Top_{num_top}_Features', index=False)
